# TMJDLC — Javtiful Download Test

This notebook tests the **Javtiful provider**, not the MissAV API. It opens the original Javtiful page, captures the playback API source, resolves relative media paths, and downloads immediately in the same Colab runtime. If a download attempt fails, the provider resolves a fresh temporary URL before retrying.

Keep the same Colab runtime/network throughout resolution and download. Do not copy, save, or share temporary signed media URLs.

In [ ]:
# Install TMJDLC and Chromium
!git clone --depth 1 https://github.com/Telebotfaroff/TMJDLC.git
%cd TMJDLC
!python -m pip install --upgrade pip
!python -m pip install -e .
!python -m playwright install --with-deps chromium
!sudo apt-get update -qq && sudo apt-get install -y ffmpeg

## 1. Configure the download

In [ ]:
#@title Javtiful download settings
VIDEO_URL = "https://javtiful.com/video/115269/miab-682-reducing-mosaic" #@param {type:"string"}
QUALITY = "best" #@param ["best", "1080", "720", "480", "360"]
OUTPUT_DIR = "/content/tmjdlc-downloads" #@param {type:"string"}
MAX_ATTEMPTS = 2 #@param {type:"integer"}

from pathlib import Path
from urllib.parse import urlsplit
if urlsplit(VIDEO_URL).hostname not in {"javtiful.com", "www.javtiful.com"}:
    raise ValueError("VIDEO_URL must be an original javtiful.com video page URL")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
quality_value = None if QUALITY == "best" else int(QUALITY)
print(f"URL: {VIDEO_URL}")
print(f"Quality: {QUALITY}")
print(f"Maximum fresh-resolution attempts: {MAX_ATTEMPTS}")

## 2. Resolve and download
The provider performs playback resolution immediately before each download attempt. Signed media URLs remain in memory only; the displayed stream report redacts sensitive query parameters.

In [ ]:
import json
import re
import time
from tmjdlc.providers.javtiful import JavtifulProvider

video_match = re.search(r"/video/(\\d+)", VIDEO_URL)
video_id = video_match.group(1) if video_match else "video"
filename = f"javtiful-{video_id}"

def report_progress(message):
    print(f"[TMJDLC] {message}")

provider = JavtifulProvider(
    max_attempts=MAX_ATTEMPTS,
    progress_callback=report_progress,
)
started = time.monotonic()
downloaded_path, stream = provider.download(
    VIDEO_URL.strip(),
    OUTPUT_DIR,
    filename,
    quality=quality_value,
)
elapsed = time.monotonic() - started
size = downloaded_path.stat().st_size
if size <= 0:
    raise RuntimeError("Downloaded file is empty")

download_result = {
    "status": "downloaded",
    "filename": downloaded_path.name,
    "size_bytes": size,
    "size_mib": round(size / (1024 ** 2), 2),
    "elapsed_seconds": round(elapsed, 2),
    "average_mib_per_second": round(size / (1024 ** 2) / elapsed, 3) if elapsed else None,
    "stream": stream.to_public_dict(),
}
print("\\nDOWNLOAD RESULT")
print(json.dumps(download_result, ensure_ascii=False, indent=2))

## 3. Validate with FFprobe

In [ ]:
import subprocess
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=format_name,duration,size:stream=codec_type,codec_name,width,height",
     "-of", "json", str(downloaded_path)],
    capture_output=True, text=True,
)
if probe.returncode != 0:
    print(probe.stderr[-4000:])
    raise RuntimeError("FFprobe could not validate the downloaded file")
probe_data = json.loads(probe.stdout)
print(json.dumps(probe_data, indent=2))

## 4. Download the completed file to your device

In [ ]:
from google.colab import files
files.download(str(downloaded_path))

## 5. Send the test results
If the run fails, share the progress output and traceback. The safe stream report masks signed query values. Never share cookies, authorization headers, or a raw signed media URL.